In [ ]:
import datetime as dt
import dask.dataframe as dd
import pandas as pd
import dask
from sqlalchemy import select, create_engine
from sqlalchemy.engine import Engine
from sqlalchemy.dialects import postgresql
from sqlalchemy.sql.expression import func
from sqlalchemy.sql.expression import literal_column, literal
from sqlalchemy.dialects.postgresql import INTERVAL
from dotenv import load_dotenv
import os
from statsmodels.tsa.stattools import coint
import mc_postgres_db.models as models
from sqlalchemy.orm import Session
from coiled import Cluster

load_dotenv()

POSTGRES_URL = os.getenv("POSTGRES_URL")

engine = create_engine(POSTGRES_URL)

In [ ]:
def fix_divisions(df):
    """
    Ensure divisions are known and properly set.
    This is THE most common fix for merge_asof issues.
    """
    if not df.known_divisions:
        df = df.reset_index().set_index("timestamp", sorted=True)
    return df


def repartition_by_size(df, partition_size="100MB"):
    """
    Repartition based on memory size rather than number of rows.
    """
    return df.repartition(partition_size=partition_size)


def align_divisions(left_df, right_df):
    """
    Make both dataframes use the same divisions.
    This ensures matching partitions can find each other.
    """
    if left_df.npartitions <= right_df.npartitions:
        divisions = left_df.divisions
        right_df = right_df.repartition(divisions=divisions)
    else:
        divisions = right_df.divisions
        left_df = left_df.repartition(divisions=divisions)

    return left_df, right_df


def prepare_for_merge_asof(left_df, right_df):
    """
    Prepare two Dask dataframes for merge_asof.
    Returns prepared dataframes ready to merge.

    Usage:
        full_frame_fixed, market_data_fixed = prepare_for_merge_asof(full_frame, market_data)
        result = dd.merge_asof(full_frame_fixed, market_data_fixed, ...)
    """
    left_df = fix_divisions(left_df)
    right_df = fix_divisions(right_df)
    left_df, right_df = align_divisions(left_df, right_df)

    return left_df, right_df

In [ ]:
cluster = Cluster(n_workers=4, container="ghcr.io/glynfinck/mc-notebooks:main")
client = cluster.client()

In [ ]:
date: dt.date = dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(days=1)
end = dt.datetime.combine(date, dt.time.min)
start = end - dt.timedelta(days=1)
start_naive = start.replace(tzinfo=None).replace(second=0, microsecond=0)
end_naive = end.replace(tzinfo=None).replace(second=0, microsecond=0)
print(f"Start: {start}, End: {end}")

In [ ]:
max_groups = 5
with Session(engine) as session:
    # Get all provider asset group id(s)
    provider_asset_group_ids = session.scalars(
        select(models.ProviderAssetGroup.id).limit(max_groups)
    ).all()
print(
    f"Provider asset group ids (count: {len(provider_asset_group_ids)}): {provider_asset_group_ids}"
)

In [ ]:
# Now select from the subquery
time_frame: dd.DataFrame = dd.read_sql_query(
    select(
        select(
            func.generate_series(
                literal_column(start_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")),
                literal_column(end_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")),
                func.cast(literal("1 minute"), INTERVAL),
            ).label("timestamp")
        ).subquery("time_frame")
    ),
    engine.url.render_as_string(hide_password=False),
    index_col="timestamp",
    bytes_per_chunk="512 MiB",
)

In [ ]:
provider_asset_group_members = dd.read_sql_table(
    "provider_asset_group_member",
    engine.url.render_as_string(hide_password=False),
    index_col="provider_asset_group_id",
    bytes_per_chunk="512 MiB",
    columns=[
        "provider_asset_group_id",
        "order",
        "provider_id",
        "from_asset_id",
        "to_asset_id",
    ],
)

In [ ]:
time_frame["key"] = 1
provider_asset_group_members["key"] = 1
full_frame = time_frame.reset_index().merge(
    provider_asset_group_members.reset_index(),
    on="key",
)
full_frame = full_frame.drop(columns=["key"])
full_frame = full_frame.set_index("timestamp")

In [ ]:
# Load with pandas
market_data_pd = pd.read_sql(
    select(
        models.ProviderAssetMarket.timestamp,
        models.ProviderAssetMarket.provider_id,
        models.ProviderAssetMarket.from_asset_id,
        models.ProviderAssetMarket.to_asset_id,
        models.ProviderAssetMarket.close,
    ).where(
        models.ProviderAssetMarket.timestamp.between(start_naive, end_naive)
    ).order_by(models.ProviderAssetMarket.timestamp),
    engine,
    index_col="timestamp",
)
# Convert to Dask with size-based partitioning
market_data = dd.from_pandas(market_data_pd, npartitions=10)
market_data = market_data.repartition(partition_size="512MB")

In [ ]:
full_market_frame = dd.merge_asof(
    full_frame,
    market_data,
    left_index=True,
    right_index=True,
    by=["provider_id", "from_asset_id", "to_asset_id"],
)

In [ ]:
# Split out the close for each order and create pairs-trading frame.
close_1 = full_market_frame.loc[
    full_frame["order"] == 1,
    ["provider_asset_group_id", "provider_id", "from_asset_id", "to_asset_id", "close"],
]
close_2 = full_market_frame.loc[
    full_frame["order"] == 2,
    ["provider_asset_group_id", "provider_id", "from_asset_id", "to_asset_id", "close"],
]
pairs_trading_frame: dd.DataFrame = dd.merge(
    close_1,
    close_2,
    on=["timestamp", "provider_asset_group_id"],
    how="inner",
    suffixes=("_1", "_2"),
)

In [ ]:
cointegration_p_values = pairs_trading_frame.groupby("provider_asset_group_id")[
    ["close_1", "close_2"]
].apply(
    lambda df: pd.Series(coint(df["close_1"], df["close_2"])[1], index=["p_value"]),
    meta={"p_value": pd.Series([], dtype=float)},
)

In [ ]:
cointegration_p_values_computed = cointegration_p_values.compute()
cointegration_p_values_computed

In [ ]:
full_frame.loc[full_frame["provider_asset_group_id"] == 1].compute()

In [ ]:
full_frame

In [ ]:
# Create subquery that generates minutely timestamps using PostgreSQL's generate_series
start_str = start_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
end_str = end_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
time_frame_subquery = select(
    func.generate_series(
        literal_column(start_str),
        literal_column(end_str),
        func.cast(literal("1 minute"), INTERVAL),
    ).label("timestamp")
).subquery("time_frame")

# Create the sub-query for all provider asset group members.
provider_asset_group_members_subquery = (
    select(
        models.ProviderAssetGroupMember.provider_asset_group_id,
        models.ProviderAssetGroupMember.order,
        models.ProviderAssetGroupMember.provider_id,
        models.ProviderAssetGroupMember.from_asset_id,
        models.ProviderAssetGroupMember.to_asset_id,
    )
    .where(
        models.ProviderAssetGroupMember.provider_asset_group_id.in_(
            provider_asset_group_ids
        )
    )
    .subquery("provider_asset_group_members")
)

# Combine the time-frame with the provider asset group members into a dask dataframe.
# Using cross join to create a complete timeframe for each asset group member
# Wrap in an additional subquery to help Dask partition correctly
full_frame_inner = (
    select(
        time_frame_subquery.c.timestamp,
        provider_asset_group_members_subquery.c.provider_asset_group_id,
        provider_asset_group_members_subquery.c.order,
        provider_asset_group_members_subquery.c.provider_id,
        provider_asset_group_members_subquery.c.from_asset_id,
        provider_asset_group_members_subquery.c.to_asset_id,
    )
    .select_from(
        time_frame_subquery.join(
            provider_asset_group_members_subquery, literal(True), isouter=False
        )
    )
    .subquery("full_frame_inner")
)

# Select from the subquery for Dask
full_frame_query = select(
    full_frame_inner.c.timestamp,
    full_frame_inner.c.provider_asset_group_id,
    full_frame_inner.c.order,
    full_frame_inner.c.provider_id,
    full_frame_inner.c.from_asset_id,
    full_frame_inner.c.to_asset_id,
).order_by(full_frame_inner.c.timestamp)

full_frame: dd.DataFrame = dd.read_sql_query(
    full_frame_query,
    engine.url.render_as_string(hide_password=False),
    index_col="timestamp",
    bytes_per_chunk="512 MiB",
)


# Get the market data Dask dataframe.
market_data: dd.DataFrame = dd.read_sql_query(
    select(
        models.ProviderAssetMarket.timestamp,
        models.ProviderAssetMarket.provider_id,
        models.ProviderAssetMarket.from_asset_id,
        models.ProviderAssetMarket.to_asset_id,
        models.ProviderAssetMarket.close,
    )
    .where(models.ProviderAssetMarket.timestamp.between(start_naive, end_naive))
    .order_by(models.ProviderAssetMarket.timestamp),
    engine.url.render_as_string(hide_password=False),
    index_col="timestamp",
    bytes_per_chunk="512 MiB",
)

# As-of join the market data with the full frame.
framed_market_data = dd.merge_asof(
    full_frame,
    market_data,
    left_index=True,
    right_index=True,
    by=["provider_id", "from_asset_id", "to_asset_id"],
)

# Split out the close for each order and create pairs-trading frame.
close_1 = framed_market_data.loc[
    full_frame["order"] == 1,
    ["provider_asset_group_id", "provider_id", "from_asset_id", "to_asset_id", "close"],
]
close_2 = framed_market_data.loc[
    full_frame["order"] == 2,
    ["provider_asset_group_id", "provider_id", "from_asset_id", "to_asset_id", "close"],
]
pairs_trading_frame: dd.DataFrame = dd.merge(
    close_1,
    close_2,
    on=["timestamp", "provider_asset_group_id"],
    how="inner",
    suffixes=("_1", "_2"),
)

In [ ]:
pairs_trading_frame.visualize()

### Optional: Installing Graphviz for Visualization

If you want to use `.visualize()` to see Dask computation graphs, install Graphviz:

**On macOS:**
```bash
brew install graphviz
```

**On Ubuntu/Debian:**
```bash
sudo apt-get install graphviz
```

**On Windows:**
Download from https://graphviz.org/download/ and add to PATH

Then install the Python package:
```bash
pip install graphviz
```


## Debugging Dask merge_asof Issues

When `merge_asof` works in pandas but not in Dask, it's usually due to:

1. **Partitioning** - Data split across partitions incorrectly
2. **Sorting** - Index not properly sorted within each partition
3. **Data alignment** - Timestamps don't overlap between dataframes
4. **Empty partitions** - Some partitions have no data

Let's debug:


In [ ]:
cointegration_p_values = pairs_trading_frame.groupby("provider_asset_group_id")[
    ["close_1", "close_2"]
].apply(
    lambda df: pd.Series(coint(df["close_1"], df["close_2"])[1], index=["p_value"]),
    meta={"p_value": pd.Series([], dtype=float)},
)
cointegration_p_values.compute()

In [ ]:
cointegration_p_values

In [ ]:
def get_pairs_trading_frame(
    start: dt.datetime, end: dt.datetime, provider_asset_group_ids: list[int], url: str
) -> pd.DataFrame:
    """
    Get the pairs trading frame for the given start and end dates and provider asset group ids.
    """

    # Create subquery that generates minutely timestamps using PostgreSQL's generate_series
    start_str = start_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
    end_str = end_naive.strftime("'%Y-%m-%d %H:%M:%S'::timestamp")
    time_frame_subquery = select(
        func.generate_series(
            literal_column(start_str),
            literal_column(end_str),
            func.cast(literal("1 minute"), INTERVAL),
        ).label("timestamp")
    ).subquery("time_frame")

    # Create the sub-query for all provider asset group members.
    provider_asset_group_members_subquery = (
        select(
            models.ProviderAssetGroupMember.provider_asset_group_id,
            models.ProviderAssetGroupMember.order,
            models.ProviderAssetGroupMember.provider_id,
            models.ProviderAssetGroupMember.from_asset_id,
            models.ProviderAssetGroupMember.to_asset_id,
        )
        .where(
            models.ProviderAssetGroupMember.provider_asset_group_id.in_(
                provider_asset_group_ids
            )
        )
        .subquery("provider_asset_group_members")
    )

    # Combine the time-frame with the provider asset group members into a dask dataframe.
    # Using cross join to create a complete timeframe for each asset group member
    full_frame: pd.DataFrame = pd.read_sql_query(
        select(
            time_frame_subquery.c.timestamp,
            provider_asset_group_members_subquery.c.provider_asset_group_id,
            provider_asset_group_members_subquery.c.order,
            provider_asset_group_members_subquery.c.provider_id,
            provider_asset_group_members_subquery.c.from_asset_id,
            provider_asset_group_members_subquery.c.to_asset_id,
        )
        .select_from(
            time_frame_subquery.join(
                provider_asset_group_members_subquery, literal(True), isouter=False
            )
        )
        .order_by(time_frame_subquery.c.timestamp),
        engine,
        index_col="timestamp",
    )

    # Get the market data Dask dataframe.
    market_data: pd.DataFrame = pd.read_sql_query(
        select(
            models.ProviderAssetMarket.timestamp,
            models.ProviderAssetMarket.provider_id,
            models.ProviderAssetMarket.from_asset_id,
            models.ProviderAssetMarket.to_asset_id,
            models.ProviderAssetMarket.close,
        )
        .where(models.ProviderAssetMarket.timestamp.between(start_naive, end_naive))
        .order_by(models.ProviderAssetMarket.timestamp),
        engine,
        index_col="timestamp",
    )

    # As-of join the market data with the full frame.
    full_frame = pd.merge_asof(
        full_frame,
        market_data,
        left_index=True,
        right_index=True,
        by=["provider_id", "from_asset_id", "to_asset_id"],
    )

    # Split out the close for each order and create pairs-trading frame.
    close_1 = full_frame.loc[
        full_frame["order"] == 1,
        [
            "provider_asset_group_id",
            "provider_id",
            "from_asset_id",
            "to_asset_id",
            "close",
        ],
    ]
    close_2 = full_frame.loc[
        full_frame["order"] == 2,
        [
            "provider_asset_group_id",
            "provider_id",
            "from_asset_id",
            "to_asset_id",
            "close",
        ],
    ]
    return pd.merge(
        close_1,
        close_2,
        on=["timestamp", "provider_asset_group_id"],
        how="inner",
        suffixes=("_1", "_2"),
    )